In [1]:
from transformers import RobertaTokenizer, RobertaModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

c:\Users\rafin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 1: Load and preprocess text
file_path = r'C:\Users\rafin\Desktop\HSIC\Ripeness\Datasets\LE_Corpus\Kiwi_VIS_Corpus.txt'

with open(file_path, 'r') as file:
    text = file.read()
    
# Clean text and standardize
text = text.replace('-', '').replace('_', '').lower()

In [3]:
print(text)

overripe
1. overripe kiwis, both the actinidia deliciosa and actinidia chinensis varieties, exhibit unique spectral characteristics in the visible range (vis: 380740 nm). hyperspectral imaging of these fruits has provided substantial insights into their material composition, quality, and ripeness stages. for instance, the spectral behavior of overripe kiwis is largely characterized by high reflectance peaks in the range of approximately 550700 nm. these reflectance peaks correlate with the decomposition of chlorophyll pigments during postharvest ripening, signifying the degradation process. 

on the contrary, keypoints of absorption are observed in the lower wavelength range, specifically around 425450 nm and 675 nm, which correspond to chlorophyll's absorption features. these absorption features diminish as ripening advances due to chlorophyll degradation, increasing the fruit's overall light reflectance. 

hyperspectral imaging helps capture these spectral variations, which are instr

In [4]:
# Step 2: Load RoBERTa model
tokenizer = RobertaTokenizer.from_pretrained('roberta-large')
model = RobertaModel.from_pretrained('roberta-large')

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Step 3: Tokenize text with proper handling
inputs = tokenizer(text, return_tensors='pt', 
                  truncation=True, 
                  padding=True,
                  max_length=512)  # Ensure consistent length

# Convert to tokens for verification
input_ids = inputs['input_ids'][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# Print first 200 tokens to verify "perfect" exists
print("Sample tokens:", tokens[:200])

Sample tokens: ['<s>', 'over', 'ri', 'pe', 'Ċ', '1', '.', 'Ġover', 'ri', 'pe', 'Ġk', 'iw', 'is', ',', 'Ġboth', 'Ġthe', 'Ġact', 'in', 'idia', 'Ġdelic', 'ios', 'a', 'Ġand', 'Ġact', 'in', 'idia', 'Ġchin', 'ensis', 'Ġvarieties', ',', 'Ġexhibit', 'Ġunique', 'Ġspectral', 'Ġcharacteristics', 'Ġin', 'Ġthe', 'Ġvisible', 'Ġrange', 'Ġ(', 'vis', ':', 'Ġ380', '740', 'Ġnm', ').', 'Ġhypers', 'pect', 'ral', 'Ġimaging', 'Ġof', 'Ġthese', 'Ġfruits', 'Ġhas', 'Ġprovided', 'Ġsubstantial', 'Ġinsights', 'Ġinto', 'Ġtheir', 'Ġmaterial', 'Ġcomposition', ',', 'Ġquality', ',', 'Ġand', 'Ġrip', 'eness', 'Ġstages', '.', 'Ġfor', 'Ġinstance', ',', 'Ġthe', 'Ġspectral', 'Ġbehavior', 'Ġof', 'Ġover', 'ri', 'pe', 'Ġk', 'iw', 'is', 'Ġis', 'Ġlargely', 'Ġcharacterized', 'Ġby', 'Ġhigh', 'Ġreflect', 'ance', 'Ġpeaks', 'Ġin', 'Ġthe', 'Ġrange', 'Ġof', 'Ġapproximately', 'Ġ550', '700', 'Ġnm', '.', 'Ġthese', 'Ġreflect', 'ance', 'Ġpeaks', 'Ġcorrelate', 'Ġwith', 'Ġthe', 'Ġdecom', 'position', 'Ġof', 'Ġchlor', 'ophy', 'll', 'Ġpig', 'ments

In [6]:
# Step 4: Get embeddings from last hidden state
with torch.no_grad():
    outputs = model(**inputs)
embeddings = outputs.last_hidden_state  # Shape: [1, seq_len, 1024]

In [ ]:
# Step 5: Improved embedding extraction with fallback
words = ["overripe", "ripe", "unripe"]
embeddings_dict = {}

for word in words:
    # Tokenize target word (handle subwords)
    word_tokens = tokenizer.tokenize(word)
    word_ids = tokenizer.convert_tokens_to_ids(word_tokens)
    
    # Find positions in input_ids
    indices = []
    for i in range(len(input_ids) - len(word_ids) + 1):
        if all(input_ids[i+j] == word_ids[j] for j in range(len(word_ids))):
            indices.extend(range(i, i+len(word_ids)))
    
    if indices:
        # Average embeddings for multi-token words
        word_embedding = embeddings[0, indices, :].mean(dim=0)
    else:
        # Fallback: embed word in isolation
        print(f"Word '{word}' not found in text - using standalone embedding")
        word_inputs = tokenizer(word, return_tensors='pt')
        with torch.no_grad():
            word_outputs = model(**word_inputs)
        word_embedding = word_outputs.last_hidden_state.mean(dim=1).squeeze()
    
    embeddings_dict[word] = word_embedding

# %%
# Verification and analysis
print("\nEmbedding shapes:")
for word, emb in embeddings_dict.items():
    print(f"{word}: {emb.shape}")

Word 'perfect' not found in text - using standalone embedding
Word 'unripe' not found in text - using standalone embedding

Embedding shapes:
overripe: torch.Size([1024])
perfect: torch.Size([1024])
unripe: torch.Size([1024])


In [8]:
# Calculate similarity matrix
similarities = cosine_similarity(
    [embeddings_dict["overripe"].numpy(), 
     embeddings_dict["perfect"].numpy(),
     embeddings_dict["unripe"].numpy()]
)

In [10]:
print("\nCosine similarity matrix:")
print("       overripe  perfect  unripe")
for i, row in enumerate(["overripe", "perfect", "unripe"]):
    print(f"{row:8} {similarities[i][0]:.3f}    {similarities[i][1]:.3f}    {similarities[i][2]:.3f}")



Cosine similarity matrix:
       overripe  perfect  unripe
overripe 1.000    0.940    0.950
perfect  0.940    1.000    0.996
unripe   0.950    0.996    1.000


In [11]:
# Save embeddings
np.save("Avocado_VIS_Embeddings_Roberta.npy", 
        {k: v.numpy() for k, v in embeddings_dict.items()})